# Laplace and Poisson Equations

Steady states need no time: $-\Delta u = f$ balances diffusion against a source. On a grid it becomes one large, sparse, symmetric linear system. This notebook builds the five-point Laplacian with `kron`, compares a sparse direct solve with conjugate gradients, and shows the heat equation settling to the Poisson solution.

In [ ]:
using SparseArrays, LinearAlgebra, Krylov, Plots
function poisson(n; L=1.0)   # -Laplacian on the n x n interior grid of [0, L]^2, u = 0 on the boundary
    h = L / (n + 1)
    T1 = spdiagm(-1 => fill(-1.0, n - 1), 0 => fill(2.0, n), 1 => fill(-1.0, n - 1)) ./ h^2
    Id = sparse(1.0I, n, n)
    return collect(h .* (1:n)), h, kron(Id, T1) + kron(T1, Id)
end

## The five-point Laplacian

With the grid values stored as a vector (x varying fastest), `kron(I, T)` differences along $x$ and `kron(T, I)` along $y$. Each row has at most five non-zeros: the point and its four neighbours.

In [ ]:
xs, h, A = poisson(4)
(unknowns = size(A, 1), nonzeros = nnz(A), symmetric = issymmetric(A), smallest_eigenvalue = minimum(eigvals(Matrix(A))))

In [ ]:
heatmap(Matrix(A) .!= 0, yflip=true, c=:grays, colorbar=false, aspect_ratio=1, title="Sparsity pattern for a 4 x 4 grid")

## A uniform source

With $f = 1$ on the unit square (uniform heating with the edges held at zero) the centre value is about $0.0737$.

In [ ]:
n = 63
xs, h, A = poisson(n)
U = reshape(A \ ones(n^2), n, n)
m = div(n + 1, 2)
display(heatmap(xs, xs, permutedims(U), aspect_ratio=1, c=:viridis, xlabel="x", ylabel="y", title="-Laplacian(u) = 1"))
(centre_value = U[m, m],)

## Boundary values: Laplace's equation

With no source and the top edge held at 1, $\Delta u = 0$ gives the steady temperature of a plate heated from one side. The known boundary values move to the right-hand side: each grid point next to the top edge gets $1/h^2$.

In [ ]:
B = zeros(n, n); B[:, n] .= 1 / h^2   # U[i, j] = u(x_i, y_j), so j = n is the row next to the top edge
V = reshape(A \ vec(B), n, n)
heatmap(xs, xs, permutedims(V), aspect_ratio=1, c=:inferno, xlabel="x", ylabel="y", title="Laplace: top edge at 1, others at 0")

## Direct against conjugate gradients

A sparse direct solver factorises $A$ once. Conjugate gradients (Hestenes and Stiefel, 1952) needs only products $Av$. Its iteration count grows like $\sqrt{\kappa}$, where the condition number $\kappa$ grows like $1/h^2$, so iterations double each time the grid is refined.

In [ ]:
map((31, 63, 127, 255)) do k
    _, hk, Ak = poisson(k)
    b = ones(k^2)
    Ak \ b; cg(Ak, b; atol=0.0, rtol=1e-8)   # compile
    td = @elapsed xd = Ak \ b
    tc = @elapsed (xc, stats) = cg(Ak, b; atol=0.0, rtol=1e-8)
    kappa = cot(pi * hk / 2)^2
    (grid = "$k x $k", unknowns = k^2, direct_seconds = round(td, sigdigits=3), cg_seconds = round(tc, sigdigits=3), cg_iterations = stats.niter,
     condition_number = round(kappa), difference = norm(xd - xc) / norm(xd))
end

## Heat flow settles to the Poisson solution

The Poisson solution is the steady state of $u_t = \Delta u + f$. Implicit Euler steps $(I + \Delta t\,A)u^{k+1} = u^k + \Delta t f$, each a sparse solve with one reused factorisation, converge to it at the rate of the slowest mode, $2\pi^2 \approx 19.7$.

In [ ]:
dt = 0.01
F = lu(I + dt * A)
steps = accumulate((w, _) -> F \ (w .+ dt), 1:100; init=zeros(n^2))
gaps = [norm(w - vec(U)) / norm(U) for w in steps]
plot(dt .* (1:100), gaps, yscale=:log10, lw=2, label="", xlabel="t", ylabel="relative gap to Poisson", title="Approach to steady state")

## Exercises

1. Put a hot spot at $(0.3, 0.6)$, $f = 100\exp(-50((x - 0.3)^2 + (y - 0.6)^2))$, and find where $u$ is largest. Is it at the source?
2. Measure the slope of `log(gaps)` above and compare with $-2\pi^2$ adjusted for the time step: $\log(1 + 2\pi^2\Delta t)/\Delta t$.
3. Check the flux balance: $h^2\sum f$ should equal the sum of $U$ over the grid points next to the boundary. Why?
4. Build the 3-D Laplacian with three `kron` terms for $n = 31$. Compare the direct solve with CG now.